# Multi-Agent SLM vs LLM Research Experiment (Google Colab Edition)
This notebook is designed to run the experiments described in the research project. It compares the performance of a single Large Language Model (LLM) against a Multi-Agent System (MAS) based on Small Language Models (SLMs), both utilizing Retrieval-Augmented Generation (RAG).


## 1. Setup and Dependencies
First, we need to install the necessary libraries for model loading and RAG.


In [ ]:
!pip install torch transformers sentence-transformers accelerate
import torch
import json
import time
import math
import os
import re
import pandas as pd
import numpy as np
from typing import Any, Callable, Dict, List, Optional, Sequence
from dataclasses import dataclass
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer


## 2. Core Implementation
We implement the core logic for the models, the RAG module, and the Multi-Agent System.


In [ ]:
@dataclass
class Generation:
    text: str = ""
    prompt_tokens: int = 0
    completion_tokens: int = 0
    latency_s: float = 0.0
    tokens_per_s: float = 0.0
    error: Optional[str] = None

    @property
    def total_tokens(self) -> int:
        return self.prompt_tokens + self.completion_tokens

class HFModel:
    def __init__(self, name: str):
        print(f"Loading model {name}...")
        self.name = name
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForCausalLM.from_pretrained(
            name, dtype=torch.float16, device_map="auto"
        )
        self.model.eval()

    def generate(self, prompt: str, temperature: float = 0.7, max_tokens: int = 512) -> Generation:
        try:
            started = time.perf_counter()
            inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            input_len = inputs["input_ids"].shape[1]
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_tokens,
                    temperature=temperature,
                    do_sample=temperature > 0,
                    pad_token_id=self.tokenizer.eos_token_id,
                )
            latency = time.perf_counter() - started
            response_text = self.tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            return Generation(
                text=response_text,
                prompt_tokens=input_len,
                completion_tokens=outputs.shape[1] - input_len,
                latency_s=latency,
                tokens_per_s=(outputs.shape[1] - input_len) / latency if latency else 0.0,
            )
        except Exception as exc:
            return Generation(error=str(exc))

class TransformersEmbedder:
    def __init__(self, name: str = "Qwen/Qwen3-Embedding-0.6B", max_length: int = 512):
        self.name = name
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(name, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True)
        self.model.eval()

    def embed(self, text: str) -> List[float]:
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=self.max_length).to(self.model.device)
        with torch.no_grad():
            outputs = self.model(**inputs)
        vector = outputs.last_hidden_state[:, -1, :].squeeze(0)
        vector = torch.nn.functional.normalize(vector, p=2, dim=-1)
        return vector.cpu().tolist()

def cosine(a, b):
    denom = math.sqrt(sum(x*x for x in a)) * math.sqrt(sum(y*y for y in b))
    return sum(x*y for x,y in zip(a,b)) / denom if denom else 0.0

class RagModule:
    def __init__(self, documents, embedder, top_k=3):
        self.documents = documents
        self.top_k = top_k
        self._embedder = embedder
        self._vectors = [embedder(doc) for doc in documents]

    def retrieve(self, query, top_k=None):
        query_vector = self._embedder(query)
        scores = [cosine(query_vector, v) for v in self._vectors]
        k = top_k or self.top_k
        best = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return "\n\n".join(self.documents[i] for i in best)

class ModelWithRag:
    def __init__(self, model, rag):
        self.model, self.rag = model, rag
    def generate(self, prompt, temperature=0.7, max_tokens=512):
        context = self.rag.retrieve(prompt)
        full_prompt = f"Relevant context:\n{context}\n\nTask:\n{prompt}" if context else prompt
        return self.model.generate(full_prompt, temperature, max_tokens)

class MultiAgentSystem:
    def __init__(self, model, rag):
        self.model, self.rag = model, rag
    def run(self, task):
        # Simplified MAS for Colab: 3 roles
        roles = ["analyst", "researcher", "formatter"]
        responses = {}
        for role in roles:
            prompt = f"{task}\n\nAct as a {role}. Provide specialized analysis."
            # We wrap model in RAG
            rag_model = ModelWithRag(self.model, self.rag)
            responses[role] = rag_model.generate(prompt)
        
        blocks = "\n".join([f"[{r}]: {res.text}" for r, res in responses.items()])
        final_prompt = f"Task: {task}\n\nExpert responses:\n{blocks}\n\nSynthesize a final coherent answer."
        final_gen = self.model.generate(final_prompt)
        return {"final": final_gen, "experts": responses}


## 3. Data Loading
Mount Google Drive or upload files to Colab. We load `tasks.json` and `kb.json`.


In [ ]:
# Assuming files are uploaded to /content/
try:
    with open('tasks.json', 'r', encoding='utf-8') as f:
        tasks = json.load(f)
    with open('kb.json', 'r', encoding='utf-8') as f:
        kb = json.load(f)
    print("Data loaded successfully!")
except FileNotFoundError:
    print("Please upload tasks.json and kb.json to the Colab file browser.")
    tasks, kb = [], []


## 4. Executing Experiments
Now we instantiate the models and run the three experimental scenarios.


In [ ]:
# Model setup
small_model = HFModel("Qwen/Qwen3.5-2B")
large_model = HFModel("Qwen/Qwen3.5-4B") # Using 4B as requested by user previously
embedder = TransformersEmbedder()
rag = RagModule(kb, embedder)

# Experiment 1: Baseline (2B vs 4B)
print("\n--- Experiment 1: Baseline ---")
for t in tasks[:2]: # Limiting to 2 for speed
    res_s = small_model.generate(t)
    res_l = large_model.generate(t)
    print(f"Task: {t}\n2B: {res_s.text}\n4B: {res_l.text}\n{'-'*40}")

# Experiment 2: RAG Augmentation
print("\n--- Experiment 2: RAG ---")
small_rag = ModelWithRag(small_model, rag)
large_rag = ModelWithRag(large_model, rag)
for t in tasks[:2]:
    res_s = small_rag.generate(t)
    res_l = large_rag.generate(t)
    print(f"Task: {t}\n2B+RAG: {res_s.text}\n4B+RAG: {res_l.text}\n{'-'*40}")

# Experiment 3: Multi-Agent System
print("\n--- Experiment 3: MAS ---")
mas = MultiAgentSystem(small_model, rag)
for t in tasks[:2]:
    res_mas = mas.run(t)
    print(f"Task: {t}\nMAS Result: {res_mas['final'].text}\n{'-'*40}")
